### Import Dependencies

In [ ]:
from google.adk.agents import Agent 
from google.adk.models.lite_llm import LiteLlm  
import os 

from google.adk.sessions import InMemorySessionService 
from google.adk.runners import Runner 
from google.adk.agents.run_config import RunConfig  
from google.genai import types 

from utils.tools import check_warehouse_availability, reserve_warehouse_items 

### ADK Agent

In [ ]:
model = LiteLlm( 
    model = "openai/gpt-4.1-mini", 
    temperature = 0.0,  
    api_key = os.getenv("OPENAI_API_KEY"), 
) 

root_agent = Agent(  
    name = "warehouse_manager_agent", 
    model = model,  
    tools = [check_warehouse_availability, reserve_warehouse_items], 
    description="A agent that can check the availability of items in the warehouses and reserver them.", 
    instruction="""You are a part of the shopping assistant that can manage available inventory in the warehouses.

    You will be given a conversation history and a list of tools, your task is to perform actions requested by the latest user query. Answer part of the query that you can answer with the available tools.

    Instructions:
    - Use names specificly provided in the available tools. Don't add any additional text to the names.
    - You can run multipple tools at once.
    - Once you get the tool results back, you might choose to performa additional tool calls.
    - Once your suggested tool calls are done, set final_answer to True.
    - Never set final_answer to True if you are suggesting tool_calls.
    - As the final answer you should return an answer to the users query in a form of actions performed.
    - You must always check the availability of the items in the warehouses before reserving them.
    - Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
    - If you cannot reserve any items, return an answer that the order cannot be reserved.
    - If you can reserve some items, return an answer that the order can be partially reserved and include the details.
    - If only partial quantity can be reserved in some warehouses, try to combinethe required quantity from different warehouses.
    - Try to reserve items from the closest warehouse to the user first if users location is provided.
    """
)

### ADK Session

In [ ]:
session_service = InMemorySessionService() 
await session_service.create_session( 
    app_name="warehouse_app", 
    user_id="user_1", 
    session_id="session_2", 
)

runner = Runner( 
    agent=root_agent, 
    app_name="warehouse_app", 
    session_service=session_service, 
) 

In [ ]:
message = types.Content( 
    role="user", 
    parts=[types.Part(text="What is the availability of B0C4DB5WGW in all of the warehouses?")],
)

In [ ]:
result = runner.run( 
    user_id="user_1", 
    session_id="session_2", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=1,
    ),
)

In [ ]:
for event in result: 
    print(event) 

In [ ]:
session_service = InMemorySessionService() 
await session_service.create_session( 
    app_name="warehouse_app", 
    user_id="user_1", 
    session_id="session_11", 
)

runner = Runner( 
    agent=root_agent, 
    app_name="warehouse_app", 
    session_service=session_service, 
) 

message = types.Content( 
    role="user", 
    parts=[types.Part(text="What is the availability of B0C4DBSWGW in all of the warehouses?")],
)

result = runner.run( 
    user_id="user_1", 
    session_id="session_11", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=3,
    ),
)

In [ ]:
for event in result: 
    print(event) 

In [ ]:
session_service = InMemorySessionService() 
await session_service.create_session( 
    app_name="warehouse_app", 
    user_id="user_1", 
    session_id="session_12", 
)

runner = Runner( 
    agent=root_agent, 
    app_name="warehouse_app", 
    session_service=session_service, 
) 

message = types.Content( 
    role="user", 
    parts=[types.Part(text="What is the availability of B0C4DBSWGW in all of the warehouses?")],
)

result = runner.run( 
    user_id="user_1", 
    session_id="session_12", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=3,
    ),
)

In [ ]:
for event in result: 
    if event.is_final_response():
        print("Final Response:")
        print(event.content.parts[0].text) 

In [ ]:
query = "Can you reserve 5 items of B0CF57H28T in the French office?" 
message = types.Content( 
    role="user", 
    parts=[types.Part(text=query)],
)

result = runner.run( 
    user_id="user_1", 
    session_id="session_2", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=3,
    ),
)

In [ ]:
result

In [ ]:
for data in result: 
    print(data)